# M07a — base-2B sampling probe (diversity + reward signal before RL)

**Thin launcher.** All logic lives in the repo behind a CLI, so this exact command
runs unchanged on Colab, Modal, or Lightning:

    python -m pii_redteam.cli probe-base --config configs/m07a_probe.yaml

What it does (spec in configs/m07a_probe.yaml, thresholds pre-committed):
rebuilds + asserts the locks, samples the BASE Qwen3.5-2B over the frozen 200-request
bank (t=1.0 and t=0.7, n=8, top_p 0.95) plus a greedy reference, validates every
sample against the output contract, scores valid ones with frozen D0 (spans +
p_target), and reports diversity + reward-signal metrics against go/no-go thresholds.
Timing pilot prints the projected total before the full run (~30-60 min expected on T4).

Artifacts land on Drive under `pii-redteam-rl/m07a-<timestamp>/`:
`samples.jsonl`, `probe_report.json` (with the EXPERIMENTS entry generated from the
report), and `plots/`. A failed probe is a valid result; it is recorded in full.

In [ ]:
import datetime
import os
import subprocess
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True, check=True).stdout.strip()
print("repo commit:", commit)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate"], check=True)

import torch

free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb >= 10, (
    f"GPU has only {free_gb:.1f} GB free - dirty runtime. Runtime -> Disconnect and"
    " delete runtime, then reopen from GitHub and Run all."
)
print(f"GPU free: {free_gb:.1f} GB - OK ({torch.cuda.get_device_name(0)})")
os.environ["PROBE_REPO_COMMIT"] = commit

In [ ]:
import os

env = dict(os.environ, PYTHONPATH="/content/pii-redteam-rl/src")
print("Running probe (timing pilot prints the projection first)...")
raise SystemExit(
    os.system("cd /content/pii-redteam-rl && PROBE_REPO_COMMIT=" + os.environ["PROBE_REPO_COMMIT"]
              + " PYTHONPATH=src python -m pii_redteam.cli probe-base --config configs/m07a_probe.yaml")
) or None

**When the run finishes:** the GO/NO-GO table and the recommendation are printed above.
Bring back `m07a-<timestamp>/probe_report.json` from Drive — its `experiments_entry`
is appended to `EXPERIMENTS.yaml` as-is, and the recommended temperature + reward
variant go into STATE.md for M08. If the reward signal fails, the report says which
component; the fallback order is: r_logit, higher temperature, more varied contexts.